In [60]:
import pandas as pd
import random
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from transformers import PreTrainedTokenizer
import os
import sys
from tensor2tensor.data_generators import text_encoder
import importlib
from typing import List, Tuple, Dict, Iterable, Optional
import inspect
import re

In [61]:
# --- runtime hygiene for notebooks ---
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")  # stop fork warnings
os.environ.setdefault("OMP_NUM_THREADS", "32")
os.environ.setdefault("MKL_NUM_THREADS", "32")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "32")

# Optional: safer multiprocessing on some setups
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

## Loading WSD annotated data

In [2]:
data = []
with open("/srv/data/latin/LatinWSD/data/semeval_wsd.data", encoding="utf-8") as f:
    for i, line in enumerate(f):
        fields = line.rstrip('\n').split('\t', 4)
        if len(fields) == 5:
            data.append(fields)
        else:
            print(f"Malformed line at {i+1}: {fields}")

import pandas as pd
semeval_wsd = pd.DataFrame(
    data,
    columns=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context']
)
semeval_wsd.head()

,lemma,sense_id,left_context,target_word,right_context
0,acerbus,III,", </line><line> ubi pudendum est ibi eos deser...",acerbior,Herculi quam illa mihi obiectast. </line><lin...
1,acerbus,I,", a nonnullis quaeritur. Beatus uero Augustinu...",acerbam,et dentes filiorum obstupuerunt;. Item: Non e...
2,acerbus,III,", cum ex aedito loco ingentem illam hominum mu...",acerbius,"? Quid tristius expectetur, etiamsi Phaethonte..."
3,acerbus,III,", cum interea quis vestrum hoc non audivit, qu...",acerbissimum,extat indicium et ad insignem memoriam turpit...
4,acerbus,III,", cum validis lucta certabat. Tum ad Syllam it...",acerbe,"in eum invectus fuisset, ad supplicium duci j..."


In [3]:
len(semeval_wsd)

2396

In [4]:
silver_inter_wsd = pd.read_csv("/srv/data/latin/LatinWSD/data/silver_align_wsd.data",
    sep="\t",
    names=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context'],
    header=None,
    encoding="utf-8"
)

In [5]:
silver_inter_wsd.sample(10)

,lemma,sense_id,left_context,target_word,right_context
842,consilium,I,quis enim aut alexandrini belli tantam moram h...,consilio,pari casu dissimili usi sumus .
1174,credo,V,"ut haec adeptus est , portendi sibi imperium",credebat,.
173,cohors,III,centurio is praetoriae,cohortis,", a galba custodiae pisonis additus , stricto ..."
797,consilium,I,nam et aquini et fabrateriae,consilia,"sunt inita de me , quae te video inaudisse , e..."
5031,senatus,I,praefectum alae de ui et rapinis reum causam in,senatu,dicere coegit .
1554,credo,VI,nullis aeque,credunt,"extis , nullum religionum capacius iudicant an..."
1646,credo,III,si quidem hercle etiam supremi promptas thensa...,credam,.
5395,senatus,I,"et res ita est firma ut debet esse , quam cons...",senatus,consulto cognoscerent .
4805,senatus,I,nihil per,senatum,", multa et magna per populum et absente populo..."
874,consilium,I,quod si catilina in urbe ad hanc diem remansis...,consiliis,"occurri atque obstiti , tamen , ut levissime d..."


In [6]:
latbert_sense_data = pd.read_csv(
    "/home/jupyter-vojta/notebooks/latin-bert/case_studies/wsd/data/latin.sense.data",
    sep="\t",
    names=['lemma', 'sense_id', 'left_context', 'target_word', 'right_context'],
    header=None,
    encoding="utf-8"
)

In [7]:
latbert_sense_data.head(10)

,lemma,sense_id,left_context,target_word,right_context
0,ab,I,caesar maturat,ab,urbe proficisci
1,ab,I,illam ( mulierem ) usque,a,mari supero romam proficisci
2,ab,I,quemadmodum ( caesar ),a,gergovia discederet
3,ab,I,protinus,a,corfinio in siciliam miserat
4,ab,I,quasi ad adulescentem,a,patre ex seleucia veniat
5,ab,I,nigidium,a,domitio capuam venisse
6,ab,I,ego te afuisse tam diu,a,nobis dolui
7,ab,I,abesse,a,domo paulisper maluit
8,ab,I,tum brutus,ab,roma aberat
9,ab,I,quot milia fundus suus abesset,ab,urbe


In [8]:
print(latbert_sense_data.sample(5))
print(semeval_wsd.sample(5))
print(silver_inter_wsd.sample(5))

       lemma sense_id                               left_context target_word  \
5652  solus1        I                                        NaN       solum   
2641    fluo       II                  cetera nasci , occidere ,      fluere   
2380    fama       II  ut vos mihi domi eritis , proinde ego ero        fama   
2464    fero        I                            quem ( florem )      ferunt   
2904  gradus        I                                     citato       gradu   

                                     right_context  
5652     unum hoc vitium affert senectus hominibus  
2641  , labi , nec diutius esse uno et eodem statu  
2380                                         foris  
2464                                terrae solutae  
2904                              in hostem ducere  
          lemma sense_id                                       left_context  \
2243    titulus        V  , et exurrexi, quia Dominus suscepit me. d. No...   
2006    senatus        I  , candidatis Caes

In [9]:
import re
import unicodedata as ud
import pandas as pd
from typing import Iterable, Optional

TEXT_COLS = ["left_context", "target_word", "right_context"]

TAG_RE = re.compile(r"<[^>]+>")

# Latin + extended letters (covers Latin diacritics). We normalize to NFC, so combining marks are folded.
LETTER = r"A-Za-zÀ-ÖØ-öø-ÿĀ-žḀ-ỿ"

# Delete these always (they cause phantom splits): ZWSP/ZWNJ/ZWJ/NO-BREAK BOM/soft hyphen
ZERO_WIDTH_RE = re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF\u00AD]")

# Delete non-letters that are *between* letters (hyphenation debris, OCR joiners)
NONLETTER_BETWEEN_LETTERS = re.compile(fr"(?<=[{LETTER}])[^\s{LETTER}]+(?=[{LETTER}])")

# Replace all other remaining non-letters with a space
NONLETTER_TO_SPACE = re.compile(fr"[^{LETTER}\s]+")

# Optional: split lower→UPPER boundaries only (kept OFF by default for safety)
LOWER_TO_UPPER_RE  = re.compile(r"(?<=[a-zà-öø-ÿā-žḀ-ỿ])(?=[A-Z])")

def sanitize_text_columns(
    df: pd.DataFrame,
    cols: Iterable[str] = TEXT_COLS,
    *,
    split_camel: bool = False,   # ← default False to avoid unexpected splits
    drop_tags: bool = True,
    collapse_ws: bool = True,
    lower: bool = False,
) -> pd.DataFrame:
    def _clean(s: Optional[str]) -> str:
        if not isinstance(s, str):
            s = "" if s is None else str(s)

        # 0) NFC normalize to fold combining marks
        s = ud.normalize("NFC", s)

        # 1) strip tags
        if drop_tags:
            s = TAG_RE.sub(" ", s)

        # 2) kill zero-width / soft hyphen outright
        s = ZERO_WIDTH_RE.sub("", s)

        # 3) delete non-letters only when sandwiched by letters (fixes BRITA­NNI​CUS → BRITANNICUS)
        s = NONLETTER_BETWEEN_LETTERS.sub("", s)

        # 4) all other non-letters become spaces (preserves word boundaries)
        s = NONLETTER_TO_SPACE.sub(" ", s)

        # 5) optional camel split (safe: only lower→UPPER)
        if split_camel:
            s = LOWER_TO_UPPER_RE.sub(" ", s)

        # 6) whitespace + optional lower
        if collapse_ws:
            s = re.sub(r"\s+", " ", s).strip()
        if lower:
            s = s.lower()
        return s

    out = df.copy()
    out[list(cols)] = out[list(cols)].fillna("")
    for c in cols:
        out[c] = out[c].map(_clean)
    return out


In [10]:
# Clean all three datasets identically
# can also add lower=True to lowercase all columns
latbert_sense_data = sanitize_text_columns(latbert_sense_data, lower=True)
semeval_wsd        = sanitize_text_columns(semeval_wsd, lower=True)
silver_inter_wsd   = sanitize_text_columns(silver_inter_wsd, lower=True)

In [11]:
silver_inter_wsd.sample(5)

,lemma,sense_id,left_context,target_word,right_context
5271,senatus,I,,senatum,corruptum esse dicunt
5460,senatus,I,non enim te puto graecos aut oscos ludos desid...,senatu,vestro spectare possis graecos ita non ames ut...
157,cohors,II,nam consules cum senatu et,cohortibus,urbanis forum capitoliumque occupauerant asser...
5454,senatus,I,addam illud etiam quod iam ego curare non debu...,senati,consulto meus inimicus quia tuus frater erat s...
4141,salus,III,deus autem rex noster ante saeculum operatus est,salutes,in medio terrae


In [12]:
print(len(latbert_sense_data), len(semeval_wsd), len(silver_inter_wsd))

8354 2396 6218


In [13]:
import pandas as pd

TEXT_COLS = ["left_context", "target_word", "right_context"]

def quick_sanity(df: pd.DataFrame, name: str = "df", max_context_len: int = 120):
    n = len(df)
    print(f"\n[{name}] rows={n}")

    # 1) empties in key fields
    empties = {c: int((df[c].astype(str).str.strip()=="").sum()) for c in ["lemma","sense_id", *TEXT_COLS]}
    print("empty counts:", empties)

    # 2) simple tag remnants check (after your cleaning this should be 0)
    tag_left = int(df["left_context"].str.contains("<|>", regex=True, na=False).sum())
    tag_right = int(df["right_context"].str.contains("<|>", regex=True, na=False).sum())
    print("tag remnants (left/right):", tag_left, tag_right)

    # 3) context lengths (words)
    L = df["left_context"].str.split().str.len()
    R = df["right_context"].str.split().str.len()
    print("left words:  median=", int(L.median()), "  p95=", int(L.quantile(0.95)))
    print("right words: median=", int(R.median()), "  p95=", int(R.quantile(0.95)))
    too_long = ((L > max_context_len) | (R > max_context_len)).sum()
    print(f"rows with very long context (> {max_context_len} words):", int(too_long))

    # 4) target check: non-empty & “lands” correctly when we rebuild tokens
    def _ok_target(row):
        lt = (row.left_context or "").split()
        t  = (row.target_word or "")
        rt = (row.right_context or "").split()
        if t == "":
            return False
        toks = lt + [t] + rt
        return len(lt) < len(toks) and toks[len(lt)] == t

    ok_mask = df.apply(_ok_target, axis=1)
    bad_targets = int((~ok_mask).sum())
    print("rows failing simple target placement:", bad_targets)

    # 5) quick label coverage
    print("unique lemmas:", df["lemma"].nunique(), " | unique sense_ids:", df["sense_id"].nunique())

    # Return a filtered view of suspicious rows (small sample)
    if bad_targets or tag_left or tag_right or any(empties.values()):
        print("\nSample of suspicious rows:")
        display(df.loc[(~ok_mask) | (df["target_word"].astype(str).str.strip()=="")].head(5)[
            ["lemma","sense_id","left_context","target_word","right_context"]
        ])

quick_sanity(latbert_sense_data, "latbert_sense_data")
quick_sanity(semeval_wsd,        "semeval_wsd")
quick_sanity(silver_inter_wsd,   "silver_inter_wsd")


[latbert_sense_data] rows=8354
empty counts: {'lemma': 0, 'sense_id': 0, 'left_context': 994, 'target_word': 0, 'right_context': 1448}
tag remnants (left/right): 0 0
left words:  median= 3   p95= 9
right words: median= 3   p95= 9
rows with very long context (> 120 words): 0
rows failing simple target placement: 0
unique lemmas: 201  | unique sense_ids: 2

Sample of suspicious rows:


,lemma,sense_id,left_context,target_word,right_context



[semeval_wsd] rows=2396
empty counts: {'lemma': 0, 'sense_id': 0, 'left_context': 0, 'target_word': 0, 'right_context': 7}
tag remnants (left/right): 0 0
left words:  median= 83   p95= 89
right words: median= 84   p95= 90
rows with very long context (> 120 words): 0
rows failing simple target placement: 0
unique lemmas: 40  | unique sense_ids: 7

Sample of suspicious rows:


,lemma,sense_id,left_context,target_word,right_context



[silver_inter_wsd] rows=6218
empty counts: {'lemma': 0, 'sense_id': 0, 'left_context': 180, 'target_word': 0, 'right_context': 388}
tag remnants (left/right): 0 0
left words:  median= 11   p95= 43
right words: median= 10   p95= 53
rows with very long context (> 120 words): 1
rows failing simple target placement: 0
unique lemmas: 33  | unique sense_ids: 7

Sample of suspicious rows:


,lemma,sense_id,left_context,target_word,right_context


In [14]:
def drop_missing_sense(df: pd.DataFrame, sense_col: str = "sense_id") -> pd.DataFrame:
    out = df.copy()
    # Normalize to string, strip, and filter
    s = out[sense_col].astype(str).str.strip()
    mask_ok = (s.ne("")) & (s.str.lower().ne("nan"))
    dropped = (~mask_ok).sum()
    if dropped:
        print(f"Dropping {dropped} rows with empty/NaN {sense_col}.")
    return out[mask_ok].reset_index(drop=True)

# Now drop rows missing sense labels
latbert_sense_data = drop_missing_sense(latbert_sense_data)
semeval_wsd        = drop_missing_sense(semeval_wsd)
silver_inter_wsd   = drop_missing_sense(silver_inter_wsd)

print(len(latbert_sense_data), len(semeval_wsd), len(silver_inter_wsd))

8354 2396 6218


In [15]:
silver_inter_wsd[~silver_inter_wsd["sense_id"].isin(["I", "II", "III", "IV", "V", "VI", "VII"])]

,lemma,sense_id,left_context,target_word,right_context


In [16]:
silver_inter_wsd.sample(10)

,lemma,sense_id,left_context,target_word,right_context
3472,licet,I,mitto quod invidiam quod pericula quod omnis m...,licuisset,subire paratissimus fueris quod denique inimic...
762,consilium,I,sed quia tanta perturbatio et confusio est rer...,consili,paenitet te et nos qui domi sumus tibi beati v...
5979,senatus,I,neque enim est meum contra ius optime meritae ...,senatus,dicere
3760,potestas,II,in consulatu sexto et septimo postquam bella c...,potestate,in senatus populique romani arbitrium transtuli
2110,fidelis,I,vir,fidelis,multum laudabitur qui autem festinat ditari no...
853,consilium,I,neque is cum roget quid loquar cogitatumst ita...,consilium,dum rursum haud placet nec pater potis videtur...
3648,nobilitas,I,mandabatque honores,nobilitatem,maiorum claritudinem militiae inlustris domi a...
810,consilium,I,quae mens eum aut quorum,consilia,a tanta gloria sibi vero etiam necessaria ac s...
3470,licet,II,neque sum admiratus hanc epistulam quam acastu...,liceat,quid sentiam
3278,imperator,I,tuae dicis inquit togae summum,imperatorem,esse cessurum


In [17]:
latbert_sense_data["wsd_source"] = "labert_wsd"
semeval_wsd["wsd_source"] = "semeval_wsd"
silver_inter_wsd["wsd_source"]  = "silver_inter_wsd"

In [66]:
latbert_sense_data.to_parquet("../data/wsd_data/labert_sense_data.parquet")
semeval_wsd.to_parquet("../data/wsd_data/semeval_wsd.parquet")
silver_inter_wsd.to_parquet("../data/wsd_data/silver_inter_wsd.parquet")

a## Preparing for training

In [38]:
DEVICE = "cpu"
MODEL_ID = "xlm-roberta-base"
tokenizer_xlmr = AutoTokenizer.from_pretrained(MODEL_ID)
model_xlmr     = AutoModel.from_pretrained(
                MODEL_ID,
                output_hidden_states=True,
                output_attentions=True
            ).to(DEVICE).eval()

base_path = "/srv/models/latin-bert"

# Initialize the tokenizer with the vocab.txt file and the encoder
vocab_file_path = "/srv/models/latin-bert/vocab.txt" # "/Users/vojtechkase/Projects/latin-bert/models/latin_bert/vocab.txt"  # Update this path
subword_tokenizer_path = "/srv/models/latin-bert/latin.subword.encoder"
# Update this path
encoder = text_encoder.SubwordTextEncoder(subword_tokenizer_path)

spec = importlib.util.spec_from_file_location(
    "latin_tokenizer",
    os.path.join(base_path, "latin_tokenizer.py")
)
latin_tokenizer = importlib.util.module_from_spec(spec)
sys.modules["latin_tokenizer"] = latin_tokenizer   # ensure it's registered
spec.loader.exec_module(latin_tokenizer)

LatinTokenizer = latin_tokenizer.LatinTokenizer
tokenizer_labert = LatinTokenizer(vocab_file_path, encoder)
model_labert = AutoModel.from_pretrained(base_path)

In [39]:
SEED = 42
random.seed(SEED)

In [40]:
# Unique lemmas in semeval
semeval_lemmas = sorted(semeval_wsd["lemma"].unique())
n_holdout = max(1, len(semeval_lemmas) // 5)
lemma_holdout = set(random.sample(semeval_lemmas, n_holdout))

print(f"Holdout lemmas ({len(lemma_holdout)}):", list(sorted(lemma_holdout)))

Holdout lemmas (8): ['adsumo', 'cohors', 'consilium', 'consul', 'credo', 'hostis', 'humanitas', 'itero']


In [41]:
def exclude_lemmas(df: pd.DataFrame, bad_lemmas: set) -> pd.DataFrame:
    return df[~df["lemma"].isin(bad_lemmas)].copy()

# v1: only LaBERT original case study data
train_v1 = exclude_lemmas(latbert_sense_data, lemma_holdout)

# v2: LaBERT + SemEval
train_v2 = pd.concat([
    exclude_lemmas(latbert_sense_data, lemma_holdout),
    exclude_lemmas(semeval_wsd,        lemma_holdout)
], ignore_index=True)

# v3: LaBERT + SemEval + newest silver
train_v3 = pd.concat([
    exclude_lemmas(latbert_sense_data, lemma_holdout),
    exclude_lemmas(semeval_wsd,        lemma_holdout),
    exclude_lemmas(silver_inter_wsd,   lemma_holdout)
], ignore_index=True)

# Gold evaluation split: the held-out portion of SemEval
semeval_holdout = semeval_wsd[semeval_wsd["lemma"].isin(lemma_holdout)].copy()

In [42]:
semeval_holdout.sample(5)

,lemma,sense_id,left_context,target_word,right_context,wsd_source
501,credo,I,diuisos in trina modos sollemnia uobis septeno...,credita,nosce azyma sola meis ponenda altaribus esse m...,semeval_wsd
361,consilium,II,cremuti libertas quamquam circumcisis quae dix...,consilium,ordinem dividendi praeparandi probandi ratione...,semeval_wsd
66,adsumo,I,eique salis paulum adiciebant decoquebantque i...,adsumere,qui simul et alant et ventrem molliant suntque...,semeval_wsd
300,cohors,II,ad preliandum taliter princeps et comes parati...,cohortem,insilivit qui iuxta eorum potentiam defendendo...,semeval_wsd
515,credo,V,nunc ego poeta fiam uiginti minas quae nunc nu...,credo,meo ita nunc per urbem solus sermoni omnibust ...,semeval_wsd


In [63]:
len(semeval_holdout)

480

In [43]:
# ========= helpers (string-only; no GreLa token dicts needed) =========
MAX_LEN = 256

def _ws(s: str) -> List[str]:
    return s.strip().split()

def encode_trunc(text: str, tokenizer, device="cpu", max_len=512):
    kwargs = {
        "text": text,
        "return_tensors": "pt",
        "truncation": True,
        "max_length": max_len,
    }
    # Check for optional argument support
    sig = inspect.signature(tokenizer.__call__)
    if "add_special_tokens" in sig.parameters:
        kwargs["add_special_tokens"] = True

    result = tokenizer(**kwargs)
    return result.to(device) if hasattr(result, "to") else result



def _tokens_and_target_idx(left: str, target: str, right: str) -> Tuple[List[str], int]:
    lt = _ws(left); rt = _ws(right)
    return lt + [target] + rt, len(lt)

# ---- XLM-R path: uses enc.word_ids() on the BatchEncoding (fast tokenizers) ----
def span_xlmr(tokenizer, tokens: List[str], target_idx: int, max_length: int = MAX_LEN) -> List[int]:
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                    padding="max_length", truncation=True, max_length=max_length)
    word_ids = enc.word_ids(0)  # <-- on enc, not tokenizer
    return [] if word_ids is None else [i for i, wid in enumerate(word_ids) if wid == target_idx]

# ---- LaBERT path: string-only; uses tokenizer.encode + build_inputs_with_special_tokens ----
def _first_occurrence(haystack: List[int], needle: List[int]) -> Optional[int]:
    """Find the first start index of 'needle' subsequence in 'haystack'; return None if not found."""
    if not needle:
        return None
    n, m = len(haystack), len(needle)
    for i in range(0, n - m + 1):
        if haystack[i:i+m] == needle:
            return i
    return None

def span_labert(tokenizer, tokens: List[str], target_idx: int, max_length: int = MAX_LEN) -> List[int]:
    # 1) subword length per whitespace token (no specials)
    wp_lens: List[int] = []
    for tok in tokens:
        try:
            ids = tokenizer.encode(tok, add_special_tokens=False)
        except TypeError:
            # if add_special_tokens is not supported, encode without it (most custom tokenizers do plain encode)
            ids = tokenizer.encode(tok)
        wp_lens.append(len(ids))

    # 2) build a full encoded input for the sentence (with specials/padding) to get seq_len for clipping
    sent_str = " ".join(tokens)
    enc = encode_trunc(sent_str, tokenizer=tokenizer, device="cpu", max_len=max_length)  # your working helper
    seq_len = int(enc["attention_mask"][0].sum().item())

    # 3) estimate left specials robustly
    try:
        core_ids = tokenizer.encode(sent_str, add_special_tokens=False)
    except TypeError:
        # if add_special_tokens kwarg not supported, try plain encode and then strip specials via build_inputs helper
        core_ids = tokenizer.encode(sent_str)  # may include specials; we’ll still handle below

    left_specials = 0
    try:
        # Prefer tokenizer's own builder if available
        built = tokenizer.build_inputs_with_special_tokens(core_ids)
        start = _first_occurrence(built, core_ids)
        left_specials = start if start is not None else 1
    except Exception:
        # Fallback: build a version with specials via encode(add_special_tokens=True) and align
        try:
            with_spec = tokenizer.encode(sent_str, add_special_tokens=True)
            start = _first_occurrence(with_spec, core_ids)
            left_specials = start if start is not None else 1
        except Exception:
            left_specials = 1  # conservative default (CLS/<s>)

    # 4) compute target subword span indices in the final sequence
    start_in_core = sum(wp_lens[:target_idx])
    k = wp_lens[target_idx] if target_idx < len(wp_lens) else 0
    start = left_specials + start_in_core
    end = start + max(0, k) - 1
    if k <= 0:
        return []
    return [i for i in range(start, end + 1) if 0 <= i < seq_len]

def add_nav_columns_after_merge(
    df,
    tok_xlmr,
    tok_labert,
    max_length: int = MAX_LEN,
    lemma_col: str = "lemma",
):
    rows = []
    for r in df.itertuples(index=False):
        # ---- Surface view ----
        tokens_surf, tidx_surf = _tokens_and_target_idx(r.left_context, r.target_word, r.right_context)
        span_x_surf = span_xlmr(tok_xlmr,  tokens_surf, tidx_surf, max_length=max_length)
        span_l_surf = span_labert(tok_labert, tokens_surf, tidx_surf, max_length=max_length)

        # ---- Lemma view (replace target by lemma; same left/right) ----
        target_lemma = re.sub(r"\d+$", "", getattr(r, lemma_col))
        tokens_lem, tidx_lem = _tokens_and_target_idx(r.left_context, target_lemma, r.right_context)
        span_x_lem = span_xlmr(tok_xlmr,  tokens_lem, tidx_lem, max_length=max_length)
        span_l_lem = span_labert(tok_labert, tokens_lem, tidx_lem, max_length=max_length)

        row_out = {
            **r._asdict(),

            # Surface view
            "tokens": tokens_surf,
            "target_idx": tidx_surf,
            "xmlr_wp_span": span_x_surf,
            "labert_wp_span": span_l_surf,
            "xmlr_wp_len": len(span_x_surf),
            "labert_wp_len": len(span_l_surf),

            # Lemma view
            "tokens_lemma": tokens_lem,
            "target_idx_lemma": tidx_lem,
            "xmlr_wp_span_lemma": span_x_lem,
            "labert_wp_span_lemma": span_l_lem,
            "xmlr_wp_len_lemma": len(span_x_lem),
            "labert_wp_len_lemma": len(span_l_lem),
        }
        rows.append(row_out)

    return pd.DataFrame(rows)

In [44]:
train_v1_proc = add_nav_columns_after_merge(train_v1, tokenizer_xlmr, tokenizer_labert)
train_v2_proc = add_nav_columns_after_merge(train_v2, tokenizer_xlmr, tokenizer_labert)
train_v3_proc = add_nav_columns_after_merge(train_v3, tokenizer_xlmr, tokenizer_labert)
semeval_holdout_proc = add_nav_columns_after_merge(semeval_holdout, tokenizer_xlmr, tokenizer_labert)

print("no-span counts — XLM-R:",
      (train_v3_proc.xmlr_wp_len==0).sum(),
      "LaBERT:",
      (train_v3_proc.labert_wp_len==0).sum())

no-span counts — XLM-R: 0 LaBERT: 0


In [69]:
semeval_holdout_proc.to_parquet("../data/wsd_data/semeval_holdout.parquet")

In [45]:
train_v2_proc.sample(10)

,lemma,sense_id,left_context,target_word,right_context,wsd_source,tokens,target_idx,xmlr_wp_span,labert_wp_span,xmlr_wp_len,labert_wp_len,tokens_lemma,target_idx_lemma,xmlr_wp_span_lemma,labert_wp_span_lemma,xmlr_wp_len_lemma,labert_wp_len_lemma
4737,pono,I,cum,posui,librum et cum me ipse coepi cogitare,labert_wsd,"[cum, posui, librum, et, cum, me, ipse, coepi,...",1,"[2, 3]",[1],2,1,"[cum, pono, librum, et, cum, me, ipse, coepi, ...",1,"[2, 3]","[1, 2]",2,2
1005,cado,I,radicitus exturbata pinus prona,cadit,,labert_wsd,"[radicitus, exturbata, pinus, prona, cadit]",4,"[10, 11]",[8],2,1,"[radicitus, exturbata, pinus, prona, cado]",4,"[10, 11]","[8, 9]",2,2
1269,cohaereo,I,omnia autem duo ad,cohaerendum,tertium aliquid anquirunt et quasi nodum vincu...,labert_wsd,"[omnia, autem, duo, ad, cohaerendum, tertium, ...",4,"[5, 6, 7, 8]","[4, 5]",4,2,"[omnia, autem, duo, ad, cohaereo, tertium, ali...",4,"[5, 6, 7, 8]","[4, 5]",4,2
7894,ut,II,est aliquid quod dominus praestare servo debeat,ut,cibaria vestiarium,labert_wsd,"[est, aliquid, quod, dominus, praestare, servo...",7,[10],[7],1,1,"[est, aliquid, quod, dominus, praestare, servo...",7,[10],[7],1,1
2314,fallo,I,qua spe possumus,falli,deus qui potuit,labert_wsd,"[qua, spe, possumus, falli, deus, qui, potuit]",3,"[6, 7]","[3, 4]",2,2,"[qua, spe, possumus, fallo, deus, qui, potuit]",3,[6],"[3, 4]",1,2
1439,consisto,II,ipsa mihi veritas manum inicit et paulisper,consistere,et commorari cogit,labert_wsd,"[ipsa, mihi, veritas, manum, inicit, et, pauli...",7,"[13, 14]",[8],2,1,"[ipsa, mihi, veritas, manum, inicit, et, pauli...",7,"[13, 14]","[8, 9]",2,2
4170,medius1,I,cum plenus fluctu,medius,foret alveus,labert_wsd,"[cum, plenus, fluctu, medius, foret, alveus]",3,"[6, 7]",[4],2,1,"[cum, plenus, fluctu, medius, foret, alveus]",3,"[6, 7]",[4],2,1
10238,voluntas,I,quem ad modum dixi ut bellum sine tumultu poss...,voluntate,quam ut pecunias in rem publicam polliceantur ...,semeval_wsd,"[quem, ad, modum, dixi, ut, bellum, sine, tumu...",81,"[119, 120]",[90],2,1,"[quem, ad, modum, dixi, ut, bellum, sine, tumu...",81,"[119, 120]",[90],2,1
4138,manus1,I,inimicorum in,manibus,mortuus est,labert_wsd,"[inimicorum, in, manibus, mortuus, est]",2,"[5, 6]",[2],2,1,"[inimicorum, in, manus, mortuus, est]",2,[5],[2],1,1
9404,pontifex,III,multae quassatae armamentisque spoliatae naves...,pontifex,eo anno mortuus in locum eius suffectus c sulp...,semeval_wsd,"[multae, quassatae, armamentisque, spoliatae, ...",85,"[163, 164, 165]",[119],3,1,"[multae, quassatae, armamentisque, spoliatae, ...",85,"[163, 164, 165]",[119],3,1


In [46]:
print(train_v2_proc.sample(5))

        lemma sense_id                                       left_context  \
8805  fidelis        I  apud me ne time uoluptas mea quid istuc est ne...   
3115    habeo       II                                                nil   
565       an1       II                 testem non mediocrem sed haud scio   
2740    fugio        I  evolat ante omnes rapido que per aera cursu ca...   
5898   specto        I                                         gaude quod   

     target_word                                      right_context  \
8805  fideliores  semper habuisti tibi quam me tamen tibi habeo ...   
3115       habeo                                          quod agam   
565           an                                        gravissimum   
2740       fugit                                                      
5898    spectant                           oculi te mille loquentem   

       wsd_source                                             tokens  \
8805  semeval_wsd  [apud, me, ne, time

In [47]:
def validate_nav(df, tokenizer, span_col, tokens_col, max_len=256):
    # 1) spans non-empty
    empty = (df[span_col].apply(len) == 0).sum()
    # 2) spans within actual (non-padded) length
    def _ok(row):
        enc = tokenizer(" ".join(row[tokens_col]), return_tensors="pt",
                        truncation=True, max_length=max_len, padding="max_length")
        L = int(enc["attention_mask"][0].sum().item())
        return all(0 <= i < L for i in row[span_col])
    within = df.apply(_ok, axis=1).mean()
    return empty, within

for span_col, tokens_col in [
    ("xmlr_wp_span", "tokens"),
    ("labert_wp_span", "tokens"),
    ("xmlr_wp_span_lemma", "tokens_lemma"),
    ("labert_wp_span_lemma", "tokens_lemma"),
]:
    empty, within = validate_nav(train_v3_proc.sample(50, random_state=0),
                                 tokenizer_xlmr if "xmlr" in span_col else tokenizer_labert,
                                 span_col, tokens_col)
    print(f"{span_col}: empty={empty}, within_ok={within:.2f}")

xmlr_wp_span: empty=0, within_ok=1.00
labert_wp_span: empty=0, within_ok=1.00
xmlr_wp_span_lemma: empty=0, within_ok=1.00
labert_wp_span_lemma: empty=0, within_ok=1.00


In [48]:
bad_labert_surface = train_v2_proc[train_v2_proc.labert_wp_len == 0].head(10)
bad_labert_lemma   = train_v2_proc[train_v2_proc.labert_wp_len_lemma == 0].head(10)

def peek_bad(df):
    cols = ["lemma","sense_id","left_context","target_word","right_context","tokens","target_idx"]
    return df[cols].to_dict(orient="records")

peek_bad(bad_labert_surface), peek_bad(bad_labert_lemma)

([], [])

## Actual training

In [49]:
import torch
device = "cpu"
amp_dtype = torch.float16
torch.backends.cuda.matmul.allow_tf32 = True  # Ampere perf boost

model_xlmr.to(device)
model_labert.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32900, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [50]:
# freeze layers and load to device
def freeze_bottom_n(model, n=2):
    # freeze embeddings (word/pos/type)
    if hasattr(model, "embeddings"):
        for p in model.embeddings.parameters():
            p.requires_grad = False
    # freeze first n encoder blocks
    enc = getattr(model, "encoder", None)
    if enc is not None and hasattr(enc, "layer"):
        for i, layer in enumerate(enc.layer):
            if i < n:
                for p in layer.parameters():
                    p.requires_grad = False

# move to GPU and freeze
model_xlmr.to(device); freeze_bottom_n(model_xlmr, n=2)
model_labert.to(device); freeze_bottom_n(model_labert, n=2)

In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss with:
      - FP32 math for stability (works under AMP),
      - dtype-aware masking (no -inf),
      - graph-safe zero when no positives exist in the batch.
    """
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.t = temperature

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        B = z.size(0)
        if B <= 1:
            # graph-safe zero
            return z.sum() * 0.0

        # Do loss math in float32 but keep the graph
        with torch.cuda.amp.autocast(enabled=False):
            z32 = F.normalize(z.to(torch.float32), dim=1)     # no .detach() !
            sim = (z32 @ z32.t()) / self.t                    # [B,B] float32

            device = z.device
            logits_mask = ~torch.eye(B, dtype=torch.bool, device=device)
            y_i = y.view(-1, 1)
            pos_mask = (y_i == y_i.t()) & logits_mask        # [B,B] bool

            # Use a dtype-aware “very negative” (avoid FP16 overflow)
            # (we're in fp32 here, but keeping guard if you change it later)
            neg_val = torch.tensor(-1e9, dtype=sim.dtype, device=device)
            sim_masked = sim.masked_fill(~logits_mask, neg_val)

            # Stable log-softmax via logsumexp
            log_prob = sim_masked - torch.logsumexp(sim_masked, dim=1, keepdim=True)  # [B,B] float32

            pos_counts = pos_mask.sum(dim=1)  # [B]
            valid = pos_counts > 0

            if not valid.any():
                # graph-safe zero
                return z.sum() * 0.0

            loss_i = torch.zeros(B, dtype=sim.dtype, device=device)
            loss_i[valid] = -(log_prob[valid] * pos_mask[valid]).sum(dim=1) / pos_counts[valid]
            loss = loss_i.mean()

        # Cast back to z dtype (keeps graph)
        return loss.to(z.dtype)

In [54]:
from torch.utils.data import Dataset, Sampler
from collections import defaultdict
import random

class WSDDataset(Dataset):
    """
    Works with surface or lemma view (toggle in collator).
    Each item keeps tokens (surface & lemma), lemma string, sense_id label.
    """
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        # label id for SupCon: combine lemma+sense_id to avoid cross-lemma positives
        keys = (df["lemma"].astype(str) + "||" + df["sense_id"].astype(str)).tolist()
        self.label2id = {}
        self.y = []
        for k in keys:
            if k not in self.label2id:
                self.label2id[k] = len(self.label2id)
            self.y.append(self.label2id[k])

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "tokens": r["tokens"],
            "tokens_lemma": r["tokens_lemma"],
            "lemma": r["lemma"],
            "sense_id": r["sense_id"],
            "y": self.y[idx],
            # we also keep contexts in case you want windowing later
            "left_context": r["left_context"],
            "right_context": r["right_context"],
            "target_word": r["target_word"],
        }

class GroupedBatchSampler(Sampler):
    """
    Samples P labels, K examples per label => batch of P*K.
    Ensures in-batch positives for SupCon. Drops last if insufficient.
    """
    def __init__(self, labels, P=8, K=4, seed=42):
        self.labels = labels
        self.P = P
        self.K = K
        self.rng = random.Random(seed)
        # index by label
        self.by_lbl = defaultdict(list)
        for i, y in enumerate(labels):
            self.by_lbl[y].append(i)
        # shuffle per label lists
        for v in self.by_lbl.values():
            self.rng.shuffle(v)
        self.lbls = list(self.by_lbl.keys())

    def __iter__(self):
        rng = self.rng
        lbls = self.lbls[:]
        rng.shuffle(lbls)
        ptrs = {y:0 for y in lbls}
        while True:
            # pick P labels that still have at least K remaining
            avail = [y for y in lbls if ptrs[y] + self.K <= len(self.by_lbl[y])]
            if len(avail) < self.P:
                break
            chosen = rng.sample(avail, self.P)
            batch = []
            for y in chosen:
                start = ptrs[y]
                batch.extend(self.by_lbl[y][start:start+self.K])
                ptrs[y] += self.K
            yield batch

    def __len__(self):
        # Lower bound estimate; not used by PyTorch strictly
        total = sum(len(v) // self.K for v in self.by_lbl.values())
        return total // self.P

In [55]:
import torch

def _pad_trunc_enc(enc, max_len: int, pad_id: int = 0):
    """
    Ensure enc['input_ids'] and enc['attention_mask'] are [1, max_len] tensors (CPU).
    Works even if tokenizer didn't return attention_mask.
    """
    ids  = enc["input_ids"]                  # [1, L] or list
    attn = enc.get("attention_mask", None)   # [1, L] or list/None

    if not torch.is_tensor(ids):
        ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0)
    if attn is None:
        attn = torch.ones_like(ids, dtype=torch.long)
    elif not torch.is_tensor(attn):
        attn = torch.tensor(attn, dtype=torch.long).unsqueeze(0)

    L = ids.shape[1]
    if L < max_len:
        pad_len = max_len - L
        ids  = torch.cat([ids,  torch.full((1, pad_len), pad_id, dtype=ids.dtype)], dim=1)
        attn = torch.cat([attn, torch.zeros((1, pad_len), dtype=attn.dtype)], dim=1)
    elif L > max_len:
        ids  = ids[:, :max_len]
        attn = attn[:, :max_len]

    enc["input_ids"] = ids
    enc["attention_mask"] = attn
    return enc


class Collator:
    """
    CPU-only collator. Builds fixed-length encodings and target spans.
    No .to('cuda') here; device transfer happens in the training loop.
    """
    def __init__(self, tokenizer, model_kind: str, use_lemma_view: bool, max_len: int = 256):
        self.tok = tokenizer
        self.kind = model_kind      # "xlmr" or "labert"
        self.use_lemma = use_lemma_view
        self.max_len = max_len

    def __call__(self, batch):
        input_ids = []
        attention_mask = []
        spans = []
        labels = []

        for ex in batch:
            tokens = ex["tokens_lemma"] if self.use_lemma else ex["tokens"]
            labels.append(ex["y"])

            lt = ex["left_context"].split()
            target_idx = len(lt)

            if self.kind == "xlmr":
                enc = self.tok(
                    tokens,
                    is_split_into_words=True,
                    return_tensors="pt",
                    truncation=True,
                    padding="max_length",
                    max_length=self.max_len,
                )
                word_ids = enc.word_ids(0)
                span = [] if word_ids is None else [i for i, w in enumerate(word_ids) if w == target_idx]

            else:  # "labert"
                # Accept both string and list inputs; we will pass string here.
                sent_str = " ".join(tokens)
                try:
                    enc = self.tok(
                        sent_str,
                        return_tensors="pt",
                        truncation=True,
                        padding="max_length",   # accept both True / "max_length"
                        max_length=self.max_len,
                    )
                except TypeError:
                    enc = self.tok(sent_str, return_tensors="pt", truncation=True, padding=True, max_length=self.max_len)

                # Hard guarantee on shape
                pad_id = getattr(self.tok, "pad_token_id", 0) or 0
                enc = _pad_trunc_enc(enc, self.max_len, pad_id=pad_id)

                # Robust span via your helper
                target_tok = ex["lemma"] if self.use_lemma else ex["target_word"]
                tokens2 = lt + [target_tok] + ex["right_context"].split()
                span = span_labert(self.tok, tokens2, target_idx, max_length=self.max_len)

            input_ids.append(enc["input_ids"])
            attention_mask.append(enc["attention_mask"])
            spans.append(span)

        input_ids = torch.cat(input_ids, dim=0)
        attention_mask = torch.cat(attention_mask, dim=0)
        labels = torch.tensor(labels, dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "spans": spans,
            "labels": labels,
        }

In [56]:
def pool_target_span(hidden_states: torch.Tensor, spans: list[list[int]]) -> torch.Tensor:
    """
    hidden_states: [B, L, D] last hidden layer
    spans: list of index lists per batch example
    returns: [B, D] mean-pooled over each span (fallback to CLS if empty)
    """
    B, L, D = hidden_states.shape
    out = []
    cls = hidden_states[:, 0, :]  # [B, D]
    for b in range(B):
        idxs = spans[b]
        if idxs:
            h = hidden_states[b, idxs, :].mean(dim=0)
        else:
            h = cls[b]
        out.append(h)
    return torch.stack(out, dim=0)

In [62]:
import os, json, math, time
import torch
from torch.utils.data import DataLoader
from torch import amp as torch_amp  # modern AMP API (only used if amp=True & device="cuda")

# ---------------------------
# freeze_bottom_n (unchanged)
# ---------------------------
def freeze_bottom_n(model, n=2):
    if hasattr(model, "embeddings"):
        for p in model.embeddings.parameters():
            p.requires_grad = False
    enc = getattr(model, "encoder", None)
    if enc is not None and hasattr(enc, "layer"):
        for i, layer in enumerate(enc.layer):
            if i < n:
                for p in layer.parameters():
                    p.requires_grad = False


# ---------------------------
# train_encoder_only (revised but backward-compatible)
# ---------------------------
def train_encoder_only(
    model, tokenizer, dataset,
    *,
    out_dir: str,
    model_kind: str,                 # "xlmr" or "labert"
    use_lemma_view: bool = True,
    P: int = 8, K: int = 4,
    epochs: int = 2,
    lr: float = 2e-5,
    max_len: int = 256,
    grad_accum: int = 1,
    amp: bool = True,
    device: str = "cuda",
    freeze_n: int = 2,
    temperature: float = 0.07,
    num_workers: int = 0,            # safest default in notebooks
    log_every: int = 50,
    autocast_dtype: torch.dtype = torch.float16,  # GPU FP16 when amp=True
    # ---- new optional, OFF by default (no behavior change) ----
    warmup_frac: float = 0.0,        # e.g., 0.1 for 10% linear warmup; 0.0 keeps old behavior
    max_grad_norm: float | None = None,  # e.g., 1.0 to enable grad clipping; None keeps old behavior
    drop_empty_in_batch: bool = True,    # guard against occasional empty spans
):
    """
    Encoder-only fine-tuning with Supervised Contrastive loss.
    - No classifier head; we optimize the encoder so target-span embeddings cluster by sense.
    - DataLoader/Collator are CPU-only to avoid CUDA-in-fork issues.
    - Tensors are moved to 'device' inside the training loop.

    Notes on optional toggles:
      * warmup_frac > 0 enables linear LR warmup.
      * max_grad_norm enables gradient clipping.
      * drop_empty_in_batch removes items with empty target spans in-batch (cheap safety).
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- Move model & freeze bottom-n ---
    model.to(device)
    freeze_bottom_n(model, n=freeze_n)
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise ValueError("No trainable parameters found (did you freeze too much?).")

    # --- Sampler & CPU-only collator ---
    sampler = GroupedBatchSampler(labels=dataset.y, P=P, K=K)
    collate = Collator(
        tokenizer,
        model_kind=model_kind,
        use_lemma_view=use_lemma_view,
        max_len=max_len,
    )  # IMPORTANT: Collator must NOT .to('cuda') anything

    dl = DataLoader(
        dataset,
        batch_sampler=sampler,
        collate_fn=collate,
        num_workers=num_workers,  # keep 0 in notebooks to avoid forking issues
        pin_memory=(device == "cuda" and num_workers > 0),
        persistent_workers=(num_workers > 0),
    )

    # --- Optim, (optional) warmup scheduler, scaler, loss ---
    optim = torch.optim.AdamW(trainable, lr=lr)

    total_steps = epochs * (len(dl) if hasattr(dl, "__len__") else 0)
    if warmup_frac and total_steps > 0:
        from torch.optim.lr_scheduler import LambdaLR
        warmup_steps = max(1, int(warmup_frac * total_steps))
        def lr_lambda(step):
            return step / warmup_steps if step < warmup_steps else 1.0
        sched = LambdaLR(optim, lr_lambda)
    else:
        sched = None

    scaler = torch_amp.GradScaler("cuda", enabled=(amp and device == "cuda"))
    crit = SupConLoss(temperature=temperature)

    model.train()
    step = 0
    loss_ema = None

    for ep in range(1, epochs + 1):
        for i, batch in enumerate(dl, 1):
            # Move batch to device HERE (not in the collator/workers)
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)
            spans          = batch["spans"]  # keep as python list

            # Optional guard: drop items with empty spans within the batch
            if drop_empty_in_batch:
                keep_idx = [j for j, s in enumerate(spans) if len(s) > 0]
                if len(keep_idx) != len(spans):
                    if len(keep_idx) == 0:
                        # Skip this batch entirely
                        continue
                    input_ids      = input_ids[keep_idx]
                    attention_mask = attention_mask[keep_idx]
                    labels         = labels[keep_idx]
                    spans          = [spans[j] for j in keep_idx]

            # Forward (device-aware autocast; disabled on CPU)
            with torch_amp.autocast(
                device_type=("cuda" if device == "cuda" else "cpu"),
                enabled=(amp and device == "cuda"),
                dtype=autocast_dtype,
            ):
                outs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=False)
                last = outs.last_hidden_state    # [B, L, D]
                z    = pool_target_span(last, spans)  # [B, D] mean over target subword span
                loss = crit(z, labels) / grad_accum

            # Backprop (AMP aware)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Step (with optional grad clipping and warmup)
            if i % grad_accum == 0:
                if scaler.is_enabled():
                    if max_grad_norm is not None:
                        scaler.unscale_(optim)
                        torch.nn.utils.clip_grad_norm_(trainable, max_grad_norm)
                    scaler.step(optim)
                    scaler.update()
                else:
                    if max_grad_norm is not None:
                        torch.nn.utils.clip_grad_norm_(trainable, max_grad_norm)
                    optim.step()
                optim.zero_grad(set_to_none=True)
                if sched is not None:
                    sched.step()

            step += 1
            loss_val = loss.item() * grad_accum
            loss_ema = loss_val if loss_ema is None else (0.95 * loss_ema + 0.05 * loss_val)
            if log_every and (step % log_every == 0):
                print(f"[ep {ep}] step {step}{f'/{total_steps}' if total_steps else ''}  "
                      f"loss={loss_val:.4f}  ema={loss_ema:.4f}")

        # (optional) save checkpoint per epoch
        ckpt_dir = os.path.join(out_dir, f"epoch_{ep}")
        os.makedirs(ckpt_dir, exist_ok=True)
        model.save_pretrained(ckpt_dir)
        try:
            tokenizer.save_pretrained(ckpt_dir)
        except Exception:
            pass  # custom tokenizer may not support save_pretrained

    # --- final save + args ---
    model.save_pretrained(out_dir)
    try:
        tokenizer.save_pretrained(out_dir)
    except Exception:
        pass
    with open(os.path.join(out_dir, "train_args.json"), "w") as f:
        json.dump({
            "model_kind": model_kind,
            "use_lemma_view": use_lemma_view,
            "P": P, "K": K,
            "epochs": epochs, "lr": lr,
            "max_len": max_len, "grad_accum": grad_accum,
            "freeze_n": freeze_n, "temperature": temperature,
            "amp": amp, "device": device,
            "warmup_frac": warmup_frac,
            "max_grad_norm": max_grad_norm,
            "drop_empty_in_batch": drop_empty_in_batch,
        }, f, indent=2)
    print(f"Saved to: {out_dir}")

In [58]:
# Build datasets from your processed frames
train_ds_v1 = WSDDataset(train_v1_proc)
train_ds_v2 = WSDDataset(train_v2_proc)
train_ds_v3 = WSDDataset(train_v3_proc)

In [230]:
_ = train_encoder_only(
    model_xlmr, tokenizer_xlmr, train_ds_v1,
    out_dir="../data/models/xmlr_wsd/v1_cpu",
    model_kind="xlmr",
    use_lemma_view=True,
    P=6, K=3,          # start a bit smaller on CPU
    epochs=2,
    lr=1e-5,           # gentler step; CPU numerics are fine but start safe
    max_len=192,       # trims compute a lot; bump later if needed
    grad_accum=2,      # simulate larger batches
    amp=False,         # <-- CPU: keep it off for now
    device="cpu",
    freeze_n=2,
    num_workers=8,     # CPU dataloading can parallelize
    log_every=50,
)

/tmp/ipykernel_553082/4237739242.py:70: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp and device == "cuda"))
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling 

[ep 1] step 50/878  loss=0.8400  ema=0.8162
[ep 1] step 100/878  loss=0.7457  ema=0.8521
[ep 1] step 150/878  loss=0.8333  ema=0.8636
[ep 1] step 200/878  loss=0.8004  ema=0.8200
[ep 1] step 250/878  loss=0.8090  ema=0.8794
[ep 1] step 300/878  loss=0.7851  ema=0.8458
[ep 1] step 350/878  loss=1.0755  ema=0.8324
[ep 1] step 400/878  loss=0.7804  ema=0.8966
[ep 2] step 450/878  loss=0.8025  ema=0.9487
[ep 2] step 500/878  loss=0.9617  ema=0.8285
[ep 2] step 550/878  loss=0.7946  ema=0.7939
[ep 2] step 600/878  loss=0.8522  ema=0.8601
[ep 2] step 650/878  loss=0.7784  ema=0.7902
[ep 2] step 700/878  loss=0.7786  ema=0.7966
[ep 2] step 750/878  loss=0.7290  ema=0.8025
[ep 2] step 800/878  loss=0.7100  ema=0.8300
[ep 2] step 850/878  loss=1.0877  ema=0.9022
Saved to: ../data/models/xmlr_wsd/v1_cpu


In [59]:
_ = train_encoder_only(
    model_labert, tokenizer_labert, train_ds_v1,
    out_dir="../data/models/labert_wsd/v1_cpu",
    model_kind="labert",
    use_lemma_view=True,
    P=6, K=3,          # start a bit smaller on CPU
    epochs=2,
    lr=1e-5,           # gentler step; CPU numerics are fine but start safe
    max_len=192,       # trims compute a lot; bump later if needed
    grad_accum=2,      # simulate larger batches
    amp=False,         # <-- CPU: keep it off for now
    device="cpu",
    freeze_n=2,
    num_workers=16,     # CPU dataloading can parallelize
    log_every=50,
)

/tmp/ipykernel_660350/405582364.py:70: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp and device == "cuda"))
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling p

[ep 1] step 50/878  loss=2.1050  ema=1.8030
[ep 1] step 100/878  loss=1.1367  ema=1.6170
[ep 1] step 150/878  loss=1.4684  ema=1.5334
[ep 1] step 200/878  loss=1.6497  ema=1.4463
[ep 1] step 250/878  loss=1.7807  ema=1.4601
[ep 1] step 300/878  loss=1.6022  ema=1.4438
[ep 1] step 350/878  loss=1.7551  ema=1.3898
[ep 1] step 400/878  loss=1.8981  ema=1.4825
[ep 2] step 450/878  loss=1.1672  ema=1.4369
[ep 2] step 500/878  loss=1.3568  ema=1.2086
[ep 2] step 550/878  loss=1.5860  ema=1.2476
[ep 2] step 600/878  loss=1.1416  ema=1.1655
[ep 2] step 650/878  loss=1.1329  ema=1.2416
[ep 2] step 700/878  loss=1.1468  ema=1.2528
[ep 2] step 750/878  loss=1.2776  ema=1.2320
[ep 2] step 800/878  loss=1.2501  ema=1.1989
[ep 2] step 850/878  loss=1.4332  ema=1.2503
Saved to: ../data/models/labert_wsd/v1_cpu


In [ ]:
_ = train_encoder_only(
    model_xlmr, tokenizer_xlmr, train_ds_v1,
    out_dir="xmlr_wsd/v1",
    epochs=2, lr=2e-5,
    P=8, K=4,           # shrink to P=6/K=3 if OOM
    grad_accum=1,
    amp=True,
    device=device
)